# CeNN-DeltaCell V3 — Qwen3.5-0.8B cellular memory ablation

**Experimental research notebook** · Targets full-attention layer **11** by default; optional layers 15/19. Compares native attention against **gated-delta only**, **fixed cellular diffusion**, **content-adaptive diffusion**, and **adaptive diffusion with bounded Local32 retrieval**. 

**Architecture:** Preserve pretrained Qwen Q/K/V projections, QK normalization, RoPE, output gate, output projection, decoder normalization, residuals, and MLP. Replace **only** the full-attention mixer with the *repository implementation* `tinycenn_lm.cenn_deltacell_v3` (not a notebook-only source copy).

**Safety and scientific scope:** no native model modification on disk; all replacements temporary; fixed-length unpadded, causal text prefill only; `use_cache=False` mandatory; no decoding-speed claims. Independent WikiText-2 `train`, `validation`, and `test`. Candidate choice happens **before** final test. No pretrained CeNN weights or GPU results are pre-populated.

The new component is chunkwise convex *memory-feature-cell* diffusion; test its contribution against fixed and zero-diffusion controls rather than assuming novelty or speedup.

## 0 · GPU setup and pinned software

Choose **Runtime → Change runtime type → NVIDIA GPU**. The pinned Transformers version matches existing Qwen3.5 experiments in TinyCeNN-LM. The notebook runs against the merged `main` repository and prints the exact Git commit for reproducibility.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"

In [ ]:
import os,sys,subprocess, pathlib, time, gc, json, hashlib, random, statistics, math, copy
import numpy as np, torch, pandas as pd, matplotlib.pyplot as plt
from torch.nn import functional as F
from dataclasses import asdict
from contextlib import contextmanager
from pathlib import Path

REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3, Qwen35DeltaCellAttention
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('source git SHA:',GIT_SHA,'torch:',torch.__version__)
assert torch.cuda.is_available(), 'The Qwen3.5 experiment requires an NVIDIA GPU runtime'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU',torch.cuda.get_device_name(0),'dtype',DTYPE)
subprocess.run(['nvidia-smi'],check=False)

## 1 · Experiment settings

**Quick** is a feasibility screen (80 optimizer steps/candidate); **research** uses longer sequences and more calibration. Neither profile alone demonstrates a statistically conclusive improvement. Select additional full-attention layers via `TARGET_LAYERS=[11,15,19]`. Do **not** replace Qwen's Gated DeltaNet layers with this full-attention adapter.

In [ ]:
# @title Experiment configuration
MODEL_ID='Qwen/Qwen3.5-0.8B'
PROFILE='quick' # @param ['quick','research']
TARGET_LAYERS=[11]  # Optional: [11, 15, 19]. Full-attention layers only.
SEQ_LEN=128 if PROFILE=='quick' else 512
CALIB_BLOCKS=32 if PROFILE=='quick' else 96
DEV_BLOCKS=16 if PROFILE=='quick' else 48
TEST_BLOCKS=24 if PROFILE=='quick' else 64
TRAIN_STEPS=80 if PROFILE=='quick' else 400
BATCH_SIZE=2
LR=4e-4
FEATURE_DIM=32
CHUNK_SIZE=32
SEED=2031
CANDIDATES={
 'delta_only':dict(mode='none',local_window=0),
 'fixed_cell':dict(mode='fixed',local_window=0),
 'adaptive_cell':dict(mode='adaptive',local_window=0),
 'adaptive_local32':dict(mode='adaptive',local_window=32),
}
OUT=Path('/content/cenn_deltacell_v3_qwen35')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Layers',TARGET_LAYERS,'candidates',list(CANDIDATES),'profile',PROFILE)

## 2 · Core numerical contract smoke tests (before downloading Qwen)

The *canonical* test suite lives at `tests/test_cenn_deltacell_v3.py`. This additional GPU smoke checks gradients and streaming when a GPU is available.

In [ ]:
torch.manual_seed(SEED)
toy=CeNNDeltaCellV3(num_heads=4,num_kv_heads=2,head_dim=8,
                    feature_dim=6,chunk_size=7,mode='adaptive',local_window=5).to(DEVICE)
q=torch.randn(1,4,23,8,device=DEVICE)
k=torch.randn(1,2,23,8,device=DEVICE)
v=torch.randn(1,2,23,8,device=DEVICE)
full,st=toy(q,k,v,return_state=True)
parts=[];s=None
for a,b in [(0,1),(1,8),(8,15),(15,23)]:
    y,s=toy(q[:,:,a:b],k[:,:,a:b],v[:,:,a:b],s,True)
    parts.append(y)
error=(full-torch.cat(parts,2)).abs().max().item()
assert error<3e-5,(error,st.position,s.position)
full.square().mean().backward()
assert toy.cell_router.weight.grad is not None and toy.cell_router.weight.grad.abs().sum()>0
print('GPU streaming max abs error:',error,'state bytes:',st.nbytes)
del toy,q,k,v,full,parts,s,st;torch.cuda.empty_cache()

## 3 · Load original Qwen text model, verify full-attention locations and freeze teacher weights

Only the **text** CausalLM is used; multimodal/vision input and autoregressive cache are outside this experiment.

In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
FULL=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert cfg.model_type=='qwen3_5_text',cfg.model_type
assert FULL==[3,7,11,15,19,23],FULL
assert all(i in FULL for i in TARGET_LAYERS), (FULL,TARGET_LAYERS)
print('Pinned Hugging Face model SHA:',REVISION,'layers:',cfg.num_hidden_layers,'full attention:',FULL)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(
    MODEL_ID,revision=REVISION,dtype=DTYPE,attn_implementation='sdpa',
    low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
for i in TARGET_LAYERS:
    native=LAYERS[i].self_attn
    print('layer',i,'q-heads',native.q_proj.out_features//(2*native.head_dim),
          'kv-heads',native.k_proj.out_features//native.head_dim,
          'head_dim',native.head_dim)
print('Total native parameters:',sum(p.numel() for p in model.parameters()))

## 4 · Independent token partitions

WikiText-2 raw train/validation/test are separate. This experiment uses **equal-length unpadded windows**, separated with the tokenizer EOS marker between source lines. The small default validation/test sizes are a screening limitation; increase them before scientific claims.

In [ ]:
from datasets import load_dataset

def windows_for_split(split, count, length):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    ids=[]
    for item in ds:
        sentence=item['text'].strip()
        if not sentence: continue
        ids.extend(tokenizer.encode(sentence,add_special_tokens=False))
        ids.append(tokenizer.eos_token_id)
        if len(ids)>=(count+1)*length: break
    if len(ids)<count*length:
        raise RuntimeError(f'Not enough text for {split}: {len(ids)} tokens')
    return torch.tensor(ids[:count*length],dtype=torch.long).reshape(count,length)
TRAIN=windows_for_split('train',CALIB_BLOCKS,SEQ_LEN)
DEV=windows_for_split('validation',DEV_BLOCKS,SEQ_LEN)
TEST=windows_for_split('test',TEST_BLOCKS,SEQ_LEN)
DATA_HASHES={name:hashlib.sha256(t.numpy().tobytes()).hexdigest() for name,t in
             [('train',TRAIN),('validation',DEV),('test',TEST)]}
assert len(set(DATA_HASHES.values()))==3
print('data',TRAIN.shape,DEV.shape,TEST.shape,'hashes',DATA_HASHES)

## 5 · Frozen teacher: capture native pre-mixer activations, original mixer outputs, and exact RoPE embeddings

Capture targets independently for **training** and **validation**; test is not touched until final audit. All original Qwen layers remain frozen. We use forward hooks *temporarily*, then remove them.

In [ ]:
TEACHER={i:{} for i in TARGET_LAYERS}

@torch.inference_mode()
def forward_native(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(split,ids):
    rows={i:{'x':[],'y':[],'cos':[],'sin':[]} for i in TARGET_LAYERS}
    hooks=[]
    for i in TARGET_LAYERS:
        def fn(module,args,kwargs,out,idx=i):
            x=kwargs['hidden_states']
            y=out[0] if isinstance(out,tuple) else out
            cos,sin=kwargs['position_embeddings']
            rows[idx]['x'].append(x.detach().cpu().to(torch.float16))
            rows[idx]['y'].append(y.detach().cpu().to(torch.float16))
            rows[idx]['cos'].append(cos.detach().cpu().to(torch.float32))
            rows[idx]['sin'].append(sin.detach().cpu().to(torch.float32))
        hooks.append(LAYERS[i].self_attn.register_forward_hook(fn,with_kwargs=True))
    try:
        for j in range(0,len(ids),BATCH_SIZE):
            forward_native(ids[j:j+BATCH_SIZE],keep=1)
    finally:
        for h in hooks: h.remove()
    for i in TARGET_LAYERS:
        TEACHER[i][split]={key:torch.cat(val,dim=0).contiguous() for key,val in rows[i].items()}
        print(split,'layer',i,'x/y',TEACHER[i][split]['x'].shape,
              'teacher_y_rms',TEACHER[i][split]['y'].float().square().mean().sqrt().item())

capture('train',TRAIN)
capture('dev',DEV)
torch.cuda.empty_cache()

## 6 · Build the **reusable repository** CeNN-DeltaCell V3 adapter

The pretrained query/key/value projections, QK norms, RoPE, output gate, and output projection remain **native and frozen**. The adapter trains only the new cellular recurrent core. For the ablation, the three nonlocal variants use identical parameter initialization on each target layer. Keeping native projections in the adapter does not establish that the replacement can be accelerated over FlashAttention; measure it.

In [ ]:
def make_adapter(idx,name):
    native=LAYERS[idx].self_attn
    config=dict(num_heads=native.q_proj.out_features//(2*native.head_dim),
                num_kv_heads=native.k_proj.out_features//native.head_dim,
                head_dim=native.head_dim,feature_dim=FEATURE_DIM,
                chunk_size=CHUNK_SIZE,**CANDIDATES[name])
    core=CeNNDeltaCellV3(**config).to(DEVICE)
    adapter=Qwen35DeltaCellAttention(native,core).to(DEVICE)
    # Train only adapter.core; native projections always frozen.
    assert all(not p.requires_grad for p in native.parameters())
    return adapter,config

@contextmanager
def graft(idx,adapter):
    original=LAYERS[idx].self_attn
    assert original is adapter.original
    LAYERS[idx].self_attn=adapter
    try: yield
    finally: LAYERS[idx].self_attn=original

def batch_capture(idx,split,index):
    d=TEACHER[idx][split]
    a=index
    # RoPE cos/sin remain float32; projections stay model dtype.
    return (d['x'][a].to(DEVICE,dtype=DTYPE),
            d['y'][a].to(DEVICE,dtype=DTYPE),
            (d['cos'][a].to(DEVICE),d['sin'][a].to(DEVICE)))

def local_measures(pred,target):
    p,t=pred.float(),target.float()
    nmse=(p-t).square().mean()/t.square().mean().clamp_min(1e-8)
    cosine=F.cosine_similarity(p,t,dim=-1).mean()
    return nmse,cosine

@torch.no_grad()
def local_validation(adapter,idx):
    adapter.eval();total=0.;energy=0.;num=0;cos_sum=0.
    for j in range(0,DEV_BLOCKS,BATCH_SIZE):
        xs=torch.arange(j,min(j+BATCH_SIZE,DEV_BLOCKS))
        x,y,pos=batch_capture(idx,'dev',xs)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=pos)
        p=pred.float();y=y.float()
        total+=(p-y).square().sum().item();energy+=y.square().sum().item()
        cos_sum+=F.cosine_similarity(p,y,dim=-1).sum().item()
        num+=p.shape[0]*p.shape[1]
    return dict(cosine=cos_sum/num,nmse=total/max(energy,1e-8))

## 7 · Calibrate each candidate on training activations; select checkpoints on validation activations only

**Primary loss:** normalized squared-error + cosine penalty. Per-layer candidate initialization is synchronized across modes so diffusion is the main intervention (the Local32 candidate adds trainable local gate). The original decoder's residual is *not* double-added.

In [ ]:
def train_adapter(idx,name,init_seed=SEED):
    torch.manual_seed(init_seed+idx)
    adapter,config=make_adapter(idx,name)
    adapter.train()
    optim=torch.optim.AdamW(adapter.core.parameters(),lr=LR,weight_decay=0.01)
    scaler=torch.amp.GradScaler('cuda',enabled=DTYPE==torch.float16)
    best=math.inf;best_sd=None;best_metrics=None
    for step in range(1,TRAIN_STEPS+1):
        sample=torch.randint(0,CALIB_BLOCKS,(BATCH_SIZE,))
        x,y,pos=batch_capture(idx,'train',sample)
        optim.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=pos)
            nmse,cos=local_measures(pred,y)
            loss=nmse+0.15*(1.-cos)
        if not bool(torch.isfinite(loss)):
            raise RuntimeError(f'Nonfinite loss {idx}/{name}/{step}')
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(adapter.core.parameters(),1.0)
        scaler.step(optim);scaler.update()
        if step==1 or step==TRAIN_STEPS or step%max(1,TRAIN_STEPS//5)==0:
            m=local_validation(adapter,idx)
            score=m['nmse']+.15*(1.-m['cosine'])
            if score<best:
                best=score;best_metrics=m
                best_sd={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
            print(f'layer={idx} {name} step={step} loss={loss.item():.5f} '
                  f'devNMSE={m["nmse"]:.4f} devCOS={m["cosine"]:.4f}',flush=True)
            adapter.train()
    assert best_sd is not None
    adapter.core.load_state_dict(best_sd)
    adapter.eval()
    return adapter,config,best_metrics,best_sd

## 8 · Evaluate **whole-model** next-token loss and prediction agreement on validation

Score several positions per window rather than one token; still a modest screening set. Native logits are cached on CPU to compare next-token KL and top-1 agreement. Prefill latency is measured using CUDA events and synchronized repeats, with the same inference configuration for native and grafted models.

In [ ]:
EVAL_POSITIONS=torch.tensor(list(range(7,SEQ_LEN-1,8)),device=DEVICE,dtype=torch.long)

@torch.inference_mode()
def evaluate_logits(ids,teacher_logits=None):
    n=0;nll=0.;gold=0;agree=0;kl=0.;ref=[]
    for start in range(0,len(ids),BATCH_SIZE):
        chunk=ids[start:start+BATCH_SIZE]
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=chunk.to(DEVICE),use_cache=False,
                         logits_to_keep=EVAL_POSITIONS).logits.float()
        truth=chunk[:,EVAL_POSITIONS.cpu()+1].to(DEVICE)
        logp=F.log_softmax(logits,-1)
        nll-=logp.gather(-1,truth.unsqueeze(-1)).sum().item()
        gold+=(logits.argmax(-1)==truth).sum().item()
        if teacher_logits is None: ref.append(logits.detach().cpu().to(torch.float16))
        else:
            truthlogits=teacher_logits[start//BATCH_SIZE].to(DEVICE,dtype=torch.float32)
            logt=F.log_softmax(truthlogits,-1)
            kl+=(logt.exp()*(logt-logp)).sum().item()
            agree+=(logits.argmax(-1)==truthlogits.argmax(-1)).sum().item()
        n+=truth.numel()
    m=dict(nll=nll/n,next_token_accuracy=gold/n,decisions=n)
    if teacher_logits is not None:
        m.update(teacher_kl=kl/n,teacher_agreement=agree/n)
    return m,ref

@torch.inference_mode()
def benchmark_prefill(ids,repeats=7,warmups=3):
    sample=ids[:min(BATCH_SIZE,len(ids))].to(DEVICE)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    measures=[]
    for run in range(warmups+repeats):
        begin=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        begin.record()
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
        end.record();torch.cuda.synchronize()
        assert bool(torch.isfinite(logits).all())
        if run>=warmups: measures.append(begin.elapsed_time(end))
    return dict(median_latency_ms=statistics.median(measures),
                peak_cuda_allocated_mib=torch.cuda.max_memory_allocated()/1024**2)

NATIVE_DEV,NATIVE_DEV_LOGITS=evaluate_logits(DEV)
NATIVE_BENCH=benchmark_prefill(DEV)
print('Native dev:',NATIVE_DEV,'native benchmark:',NATIVE_BENCH)

## 9 · Run the controlled cellular ablation

Candidate checkpoints contain **only new core parameters**. Native pretrained projections are recovered from the same pinned Qwen model. All candidates have the same training examples, optimizer budget, and base pretrained weights. The `delta_only` vs `fixed_cell` vs `adaptive_cell` comparison isolates the cellular transition, while `adaptive_local32` tests precise local retrieval.

In [ ]:
REPORT=[]
for idx in TARGET_LAYERS:
    for name in CANDIDATES:
        print(f'\n=== LAYER {idx}, {name} ===',flush=True)
        adapter,config,loc,state=train_adapter(idx,name)
        with graft(idx,adapter):
            dev,_=evaluate_logits(DEV,teacher_logits=NATIVE_DEV_LOGITS)
            bench=benchmark_prefill(DEV,repeats=5,warmups=2)
        row=dict(layer=idx,variant=name,**config,
                 train_steps=TRAIN_STEPS,trainable_core_params=sum(p.numel() for p in adapter.core.parameters()),
                 local_dev_nmse=loc['nmse'],local_dev_cosine=loc['cosine'],
                 dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
                 dev_agreement=dev['teacher_agreement'],dev_teacher_kl=dev['teacher_kl'],
                 val_decisions=dev['decisions'],
                 prefill_ms=bench['median_latency_ms'],
                 prefill_speedup=NATIVE_BENCH['median_latency_ms']/bench['median_latency_ms'],
                 peak_gpu_mib=bench['peak_cuda_allocated_mib'])
        REPORT.append(row)
        torch.save({'core_state':state,'core_config':config,'layer':idx,'variant':name,
                    'model_id':MODEL_ID,'model_sha':REVISION,'git_sha':GIT_SHA,'use_cache':False},
                   OUT/'checkpoints'/f'v3_layer{idx}_{name}.pt')
        pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
        print(json.dumps(row,indent=2),flush=True)
        del adapter,state
        gc.collect();torch.cuda.empty_cache()
print('Candidates completed',len(REPORT))

## 10 · Validation-only gates and selection

**Do not select fastest broken model.** Gate: local cosine ≥0.95; validation ΔNLL ≤+0.15 nats; teacher agreement ≥0.90. If no variant passes, mark everything rejected and carry the best candidate into *diagnostic-only* held-out testing. A GPU speedup is additionally required on final test to claim a pass.

In [ ]:
df=pd.DataFrame(REPORT)
MAX_DELTA_NLL=.15
MIN_COSINE=.95
MIN_AGREEMENT=.90
df['quality_gate']=((df.local_dev_cosine>=MIN_COSINE)&
                    (df.dev_delta_nll<=MAX_DELTA_NLL)&
                    (df.dev_agreement>=MIN_AGREEMENT))
df=df.sort_values(['quality_gate','dev_delta_nll','dev_teacher_kl'],ascending=[False,True,True])
display(df[['layer','variant','local_dev_cosine','local_dev_nmse','dev_delta_nll',
            'dev_agreement','prefill_speedup','quality_gate']])
passing=df[df.quality_gate]
if len(passing):
    win=passing.sort_values(['prefill_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
    eligible=True
else:
    win=df.sort_values(['dev_delta_nll','dev_teacher_kl']).iloc[0]
    eligible=False
print('Selected for diagnostic/audit:',int(win.layer),win.variant,'quality accepted:',eligible)
plt.figure(figsize=(10,4))
labels=[f'{row.layer}:{row.variant}' for row in df.itertuples()]
plt.bar(range(len(df)),df.local_dev_cosine)
plt.axhline(MIN_COSINE,linestyle='--',label='Required local cosine')
plt.xticks(range(len(df)),labels,rotation=75)
plt.ylabel('Local teacher cosine');plt.legend();plt.tight_layout();plt.show()
df.to_csv(OUT/'validation_ranked.csv',index=False)

## 11 · One held-out TEST evaluation AFTER selection

The test split was not used to tune architectures, checkpoints, or thresholds. Save both original and selected-model metrics. **Do not publish this as a quality-preserving replacement unless both validation and test gates pass.**

In [ ]:
WIN_LAYER=int(win.layer)
WIN_NAME=str(win.variant)
payload=torch.load(OUT/'checkpoints'/f'v3_layer{WIN_LAYER}_{WIN_NAME}.pt',
                   map_location='cpu',weights_only=True)
selected,_=make_adapter(WIN_LAYER,WIN_NAME)
selected.core.load_state_dict(payload['core_state'])
selected.eval()
NATIVE_TEST,TEST_REF=evaluate_logits(TEST)
NATIVE_TEST_TIMING=benchmark_prefill(TEST)
with graft(WIN_LAYER,selected):
    STUDENT_TEST,_=evaluate_logits(TEST,teacher_logits=TEST_REF)
    STUDENT_TEST_TIMING=benchmark_prefill(TEST)
TEST_DELTA=STUDENT_TEST['nll']-NATIVE_TEST['nll']
TEST_SPEEDUP=NATIVE_TEST_TIMING['median_latency_ms']/STUDENT_TEST_TIMING['median_latency_ms']
FINAL_PASS=bool(eligible and TEST_DELTA<=MAX_DELTA_NLL
                and STUDENT_TEST['teacher_agreement']>=MIN_AGREEMENT
                and TEST_SPEEDUP>1.0)
SUMMARY=dict(model=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,profile=PROFILE,
  architecture='CeNN-DeltaCell V3',selected_layer=WIN_LAYER,selected_variant=WIN_NAME,
  other_candidates=REPORT,validation_accepted=eligible,final_pass=FINAL_PASS,
  validation_native=NATIVE_DEV,validation_baseline_latency=NATIVE_BENCH,
  test_native=NATIVE_TEST,test_student=STUDENT_TEST,
  test_native_latency=NATIVE_TEST_TIMING,test_student_latency=STUDENT_TEST_TIMING,
  test_delta_nll=TEST_DELTA,test_prefill_speedup=TEST_SPEEDUP,
  quality_thresholds=dict(max_delta_nll=MAX_DELTA_NLL,min_cosine=MIN_COSINE,
                          min_agreement=MIN_AGREEMENT),
  data_split_fingerprints=DATA_HASHES,
  limitations='Text-only; no cache, no padding, no fused kernel, no autoregressive speed test; '
              'small prefill samples. Novelty and generalization unverified.')
(OUT/'final_audit.json').write_text(json.dumps(SUMMARY,indent=2,allow_nan=False))
print(json.dumps({key:SUMMARY[key] for key in
     ['selected_layer','selected_variant','validation_accepted','final_pass',
      'test_native','test_student','test_delta_nll','test_prefill_speedup']},indent=2))
if not FINAL_PASS: print('REJECT: retain native Qwen weights and native full attention.')

## 12 · Export reproducible diagnostics and compare variants

Save the complete experiment and a compact CSV (parameters, activation fidelity, end-to-end NLL, prediction agreement, CUDA prefill). Restore all Qwen layers to their original native state after each patch.

In [ ]:
import shutil
pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4))
for name,group in pd.DataFrame(REPORT).groupby('variant'):
    ax.scatter(group['local_dev_cosine'],group['dev_delta_nll'],label=name,s=70)
ax.axvline(MIN_COSINE,linestyle='--',color='gray')
ax.axhline(MAX_DELTA_NLL,linestyle='--',color='gray')
ax.set(xlabel='Local teacher cosine (higher better)',ylabel='Validation ΔNLL (lower better)',
       title='CeNN-DeltaCell V3: fidelity vs quality')
ax.legend();fig.tight_layout();fig.savefig(OUT/'fidelity_vs_quality.png',dpi=150)
shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Report:',OUT/'final_audit.json')
print('Archive:',str(OUT)+'.zip')
from IPython.display import display
from IPython.display import FileLink
display(FileLink(str(OUT)+'.zip'))

### Scientific interpretation and follow-up

- **Contribution hypothesis:** learnable content-dependent, convex neighborhood transitions applied to a persistent **feature-memory cell grid** at global chunk boundaries.
- **Critical ablation:** `delta_only` vs `fixed_cell` vs `adaptive_cell` with identical Qwen projections and initialization; `adaptive_local32` adds local retrieval.
- **Do not overclaim:** the triangular-solve implementation is a Python/PyTorch research reference; `torch.compile` or a fused Triton kernel may be needed before GPU speed can beat native attention. No layer replacements are accepted unless explicit held-out gates pass.
- **Before publication:** add PDelta3-GDN2-CLVR+Local32 as a matched baseline, longer context (1k–8k), multiple seeds, long-range retrieval, and cached decoding; report training costs, wall clock and the statistical uncertainty of speedups.
- **Code:** [`src/tinycenn_lm/cenn_deltacell_v3.py`](../src/tinycenn_lm/cenn_deltacell_v3.py) · [Architecture](../docs/CENN_DELTACELL_V3.md) · [Unit tests](../tests/test_cenn_deltacell_v3.py).